# SynthWorlds-SM R1 multilingual SFT and unlearning pilot — v4 exploratory follow-up

This Kaggle notebook reruns the Llama 3.2 1B feasibility experiment on a versioned corpus that preserves the frozen v3 evaluation split and adds source-verified Type-B three-edge practice chains. Translations are AI-only automated consistency checked; they have not received fluent-human review or independent semantic adjudication. The 8B multi-seed study remains separate.

Run the preflight in Kaggle's interactive Notebook Editor with the `HF_TOKEN` secret attached. Kaggle API-pushed batch runs do not receive Notebook Secrets. Gold labels remain separate from model prompts and are opened only during scoring. Expensive training switches default to false.

## 1. Kaggle settings

In Notebook Settings select a GPU accelerator and enable Internet. Under Add-ons → Secrets, add a secret labeled `HF_TOKEN` containing a Hugging Face read token. First request/accept access to the gated Llama model on Hugging Face (instructions are in the project README). Run the preflight cells from the Kaggle notebook editor session. `kaggle kernels push` cannot attach notebook Secrets to its batch run. Never paste the token into notebook code.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Avoid Kaggle T4x2 DataParallel OOM; train reproducibly on one T4.
from pathlib import Path, PurePosixPath
import hashlib, shutil, stat, subprocess, sys, tarfile, zipfile

INPUT = Path('/kaggle/input')
EXTRACT_ROOT = Path('/kaggle/working/synthworlds_r1_data')
archive_paths = sorted({*INPUT.rglob('*.zip'), *INPUT.rglob('*.tar'), *INPUT.rglob('*.tar.gz'), *INPUT.rglob('*.tgz')})
if archive_paths:
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True)
    for archive_path in archive_paths:
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path) as archive:
                for member in archive.infolist():
                    relative = PurePosixPath(member.filename)
                    mode = member.external_attr >> 16
                    if (relative.is_absolute() or '..' in relative.parts
                            or stat.S_ISLNK(mode)):
                        raise RuntimeError(f'Unsafe path in SFT dataset ZIP: {member.filename}')
                archive.extractall(EXTRACT_ROOT)
        else:
            with tarfile.open(archive_path, 'r:*') as archive:
                members = archive.getmembers()
                for member in members:
                    relative = PurePosixPath(member.name)
                    if (relative.is_absolute() or '..' in relative.parts
                            or not (member.isfile() or member.isdir())):
                        raise RuntimeError(f'Unsafe path or file type in SFT dataset TAR: {member.name}')
                archive.extractall(EXTRACT_ROOT, members=members)
    matches = list(EXTRACT_ROOT.rglob('scripts/train_sft_lora.py'))
else:
    matches = list(INPUT.rglob('scripts/train_sft_lora.py'))
if len(matches) != 1:
    raise RuntimeError(f'Expected exactly one SFT project in the attached dataset; found {len(matches)}')
PROJECT_ROOT = matches[0].parent.parent
CORPUS = PROJECT_ROOT / 'sft_training' / 'corpus.jsonl'
MANIFEST = PROJECT_ROOT / 'sft_training' / 'manifest.json'
RUN_DIR = Path('/kaggle/working/sft_lora_output')
print('Project root:', PROJECT_ROOT)
print('Approved corpus present:', CORPUS.is_file() and MANIFEST.is_file())
print('Dataset archive extraction:', 'PASS' if archive_paths else 'direct files')


In [ ]:
# Kaggle images may include incompatible optional torchao; this experiment uses standard PEFT LoRA.
req = PROJECT_ROOT / 'requirements-sft.txt'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
uninstall = subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True, text=True)
if uninstall.returncode != 0:
    raise RuntimeError('Could not remove incompatible optional torchao package: ' + uninstall.stderr[-1000:])
smoke_code = "import torch; from peft import LoraConfig, get_peft_model; m=get_peft_model(torch.nn.Sequential(torch.nn.Linear(4,4)), LoraConfig(r=2,lora_alpha=4,target_modules=['0'])); assert any('lora_A' in n for n,_ in m.named_parameters()); print('PASS')"
smoke = subprocess.run([sys.executable, '-c', smoke_code], capture_output=True, text=True)
print('Optional torchao removed:', uninstall.returncode == 0)
print('Fresh-process PEFT LoRA smoke:', smoke.stdout.strip() or smoke.stderr[-2000:])
if smoke.returncode != 0:
    raise RuntimeError('PEFT LoRA smoke test failed in a fresh process.')


In [ ]:
import os, torch, transformers
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Visible CUDA GPUs:', torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Settings, then rerun this notebook.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
# Diagnose Kaggle secret access without printing the credential.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

HF_ACCESS_OK = False
HF_ACCESS_ERROR = None
HF_TOKEN = None
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN secret retrieved:', bool(HF_TOKEN))
except Exception as exc:
    HF_ACCESS_ERROR = 'secret_unavailable'
    print('HF_TOKEN secret retrieval failed:', type(exc).__name__)
    print('Resolution: run this notebook from Kaggle.com in the interactive Notebook Editor and enable Add-ons → Secrets → HF_TOKEN for this notebook.')
    print('Kaggle CLI/API kernels push jobs do not receive notebook Secrets. Do not paste the token into code or a dataset.')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    try:
        model_info = HfApi().model_info('meta-llama/Llama-3.2-1B-Instruct', token=HF_TOKEN)
        HF_ACCESS_OK = True
        print('Gated Llama model access verified:', model_info.id)
    except Exception as exc:
        HF_ACCESS_ERROR = 'gated_model_access_failed'
        print('HF model access check failed after secret retrieval:', type(exc).__name__)
        print('Check that this token belongs to the Hugging Face account approved for meta-llama/Llama-3.2-1B-Instruct and has read access.')
else:
    HF_ACCESS_ERROR = 'secret_missing_or_not_attached'
    print('HF_TOKEN is empty or not attached. Enable the secret in Add-ons → Secrets, then rerun this cell in the interactive editor.')


In [ ]:
# Corpus preflight. The combined training gate also requires successful HF model access above.
import json
if not CORPUS.is_file() or not MANIFEST.is_file():
    raise FileNotFoundError('Attached dataset is missing the SFT corpus or manifest.')
runner = PROJECT_ROOT / 'scripts' / 'train_sft_lora.py'
preflight_run = subprocess.run([sys.executable, str(runner), '--corpus', str(CORPUS), '--manifest', str(MANIFEST), '--output-dir', str(RUN_DIR)], check=True, text=True, capture_output=True)
print(preflight_run.stdout)
PREFLIGHT = json.loads(preflight_run.stdout)
CAN_TRAIN = (PREFLIGHT.get('status') == 'READY_TO_TRAIN' and
             PREFLIGHT.get('training_packages_ready') is True and HF_ACCESS_OK)
print('Training packages:', 'READY' if PREFLIGHT.get('training_packages_ready') else 'MISSING')
if HF_ACCESS_ERROR:
    print('Hugging Face access blocker:', HF_ACCESS_ERROR)
print('Combined training gate:', 'READY_TO_TRAIN' if CAN_TRAIN else 'BLOCKED — see the specific preflight errors above')


In [ ]:
# Default off to prevent accidental duplicate SFT runs. Training is fail-closed on corpus integrity and gated-model access.
RUN_TRAINING = False
if RUN_TRAINING:
    if not HF_TOKEN:
        raise RuntimeError('HF_TOKEN is unavailable. Attach it under Kaggle Add-ons → Secrets and run in the interactive Kaggle Notebook Editor; CLI/API batch jobs cannot receive notebook Secrets.')
    if not HF_ACCESS_OK:
        raise RuntimeError('HF_TOKEN was retrieved, but gated-model access failed. Accept the Llama repository terms for this Hugging Face account and verify the token has read access.')
    if PREFLIGHT.get('status') != 'READY_TO_TRAIN':
        raise RuntimeError('Corpus/package preflight is not READY_TO_TRAIN: ' + repr(PREFLIGHT.get('errors', [])))
    subprocess.run([sys.executable, str(runner), '--corpus', str(CORPUS), '--manifest', str(MANIFEST), '--output-dir', str(RUN_DIR), '--train'], check=True)
    print('Training output:', RUN_DIR)
else:
    print('Training not started. Combined training gate:', 'READY_TO_TRAIN' if CAN_TRAIN else 'BLOCKED')


## 3. English-only unlearning and multilingual evaluation

The stages below implement the R1 pilot: full five-language SFT, an English-only retain-set reference, and GA/GD/NPO updates on English forget rows. Evaluation supplies only translated questions to the model; gold labels are opened by the scoring stage after generation. Holdout examples are never used for SFT or unlearning.

All run switches remain `False` until the preflights have visibly passed. The evaluator requires a complete four-language AI-only reviewed prompt set. This is clearly labeled AI-only and is not human linguistic validation.


In [ ]:
# Default off to prevent accidental duplicate runs. Enable only stages you intentionally want to run; each consumes Kaggle GPU time.
RUN_RETAIN_ONLY = False
RUN_UNLEARNING = False
RUN_EVALUATION = False
METHODS = ("ga", "gd", "npo")
RETAIN_DIR = Path('/kaggle/working/r1_retain_only')
UNLEARN_ROOT = Path('/kaggle/working/r1_unlearning')
EVAL_ROOT = PROJECT_ROOT / 'evaluation_dataset'
EVAL_OUT = Path('/kaggle/working/r1_diagnostics')

EVAL_MANIFEST = EVAL_ROOT / 'manifest.json'
if not EVAL_MANIFEST.is_file():
    raise FileNotFoundError('R1 evaluation dataset is absent from the attached package.')
eval_manifest = json.loads(EVAL_MANIFEST.read_text())
EVAL_READY = (eval_manifest.get('evaluation_ready') is True and
              eval_manifest.get('ai_review_status') == 'AI_ONLY_REVIEWED_COMPLETE' and
              eval_manifest.get('human_review_status') == 'NOT_PERFORMED')
print('AI-only translation review:', eval_manifest.get('ai_review_status'))
print('Human review:', eval_manifest.get('human_review_status'))
print('Evaluation gate:', 'READY' if EVAL_READY else 'BLOCKED')


### 3.1 Retain-only reference checkpoint

This separate adapter starts from the same base model and trains only on the English plus translated retain split rows selected by `--include-splits retain`. It is a comparison checkpoint; it does not receive forget or holdout rows.


In [ ]:
if RUN_RETAIN_ONLY:
    if not CAN_TRAIN:
        raise RuntimeError('Hugging Face access, GPU, or corpus preflight is not ready.')
    subprocess.run([sys.executable, str(runner), '--corpus', str(CORPUS), '--manifest', str(MANIFEST),
                    '--output-dir', str(RETAIN_DIR), '--include-splits', 'retain', '--train'], check=True)
else:
    print('Retain-only training not started.')


### 3.2 English-only GA, GD, and NPO

Each method begins from the same full-SFT adapter. Forget and retain updates use English examples only. GA maximizes forget-answer NLL; GD combines this with a retain-answer CE regularizer; NPO compares forget likelihood with a frozen SFT reference and adds the same retain regularizer. The script refuses any holdout use and reports the exact configuration.


In [ ]:
UNLEARN_RUNS = {}
if RUN_UNLEARNING:
    raise RuntimeError('This legacy unlearning path is retired. Use the composition stage, which requires a passing bilingual baseline gate and a checksum-bound target-ID manifest.')
else:
    print('Legacy unlearning is disabled; use the baseline-gated composition stage below.')


### 3.3 Baseline and post-unlearning evaluation

Generation is question-only, greedy, and runs before the label file is read. Exact normalized answer matching is used; generation failures count as incorrect. The report includes full-set performance, performance on cases both answered correctly by SFT, entity-group bootstrap intervals, the paired language-minus-English difference-in-differences on bilingual precompetent cases, retain/holdout controls, and DAP. Strict normalized exact-answer matching is used.


In [ ]:
if RUN_EVALUATION:
    if not EVAL_READY:
        raise RuntimeError('Evaluation is blocked until all four translations are complete and pass AI-only review import.')
    sft_run_manifest = json.loads((RUN_DIR/'run_manifest.json').read_text())
    checkpoint_args = ['--checkpoint', 'pretrained=base', '--checkpoint', f"sft={sft_run_manifest['adapter_dir']}"]
    if not RETAIN_DIR.joinpath('run_manifest.json').is_file():
        raise FileNotFoundError('Train the retain-only reference first.')
    retain_run = json.loads(RETAIN_DIR.joinpath('run_manifest.json').read_text())
    checkpoint_args += ['--checkpoint', f"retain_only={retain_run['adapter_dir']}"]
    for method in METHODS:
        run_file = UNLEARN_ROOT / method / 'run_manifest.json'
        if not run_file.is_file():
            raise FileNotFoundError(f'Missing unlearning output for {method}.')
        run = json.loads(run_file.read_text())
        checkpoint_args += ['--checkpoint', f"{method.upper()}={run['adapter_dir']}"]
    evaluator = PROJECT_ROOT/'scripts'/'evaluate_crosslingual_unlearning.py'
    subprocess.run([sys.executable, str(evaluator), '--eval-root', str(EVAL_ROOT), '--output-dir', str(EVAL_OUT), *checkpoint_args], check=True)
    print('Diagnostic report:', EVAL_OUT/'diagnostic_report.md')
else:
    print('Evaluation not started.')


## Outputs and interpretation

Keep the baseline predictions, scores, diagnostics, and run manifests together. The frozen evaluation contains 77 forget, 26 retain, and 26 holdout chains. This is an exploratory 1B feasibility run, not the planned powered 8B main study. Translation checks are automated and AI-only; they do not establish native-speaker fluency or semantic equivalence.

## Outputs

The adapter and run manifest are written under `/kaggle/working/r1_composition_sft_disjoint`; the retain-only adapter is under `/kaggle/working/r1_composition_retain_only_disjoint`. Download both with the baseline diagnostic artifacts. This remains a 1B feasibility pilot, not the 8B multi-seed study.

The v4 corpus preserves the previous 50 source-audited Type-A train-only chains and adds 85 Type-B three-edge train-only chains. The new Type-B extension has four English prompts per chain (Q1, Q2, Q3, and full composition), translated into French, German, Hindi, and Thai with protected entity names and same-family back-translation checks. An independent validator recomputes source facts, IDs, answer mapping, and evaluation disjointness. Translation checks are automated AI-only checks; no fluent-human review or independent semantic adjudication was performed. The full-SFT run uses `forget`, `retain`, and `train_only`; the retain-only reference uses `retain` and `train_only`. Unlearning remains gated on the same predeclared bilingual-competence floor.

In [ ]:
# Composition stage controls. Keep each stage disabled until its displayed preflight passes.
RUN_COMPOSITION_SFT = False
RUN_COMPOSITION_RETAIN_ONLY = False
RUN_COMPOSITION_UNLEARNING = False
RUN_COMPOSITION_BASELINE_EVALUATION = False
RUN_COMPOSITION_POST_EVALUATION = False
COMPOSITION_BASELINE_GATE = False
COMPOSITION_BASELINE_COUNTS = {}
COMPOSITION_CORPUS = PROJECT_ROOT / 'sft_training' / 'composition_corpus_disjoint.jsonl'
COMPOSITION_MANIFEST = PROJECT_ROOT / 'sft_training' / 'composition_disjoint_manifest.json'
COMPOSITION_RUN_DIR = Path('/kaggle/working/r1_composition_sft_disjoint')
COMPOSITION_RETAIN_DIR = Path('/kaggle/working/r1_composition_retain_only_disjoint')
COMPOSITION_UNLEARN_ROOT = Path('/kaggle/working/r1_composition_unlearning_disjoint')
COMPOSITION_EVAL_OUT = Path('/kaggle/working/r1_composition_diagnostics_disjoint')
COMPOSITION_TARGET_IDS = COMPOSITION_EVAL_OUT / 'precompetent_english_forget_targets.json'

if not COMPOSITION_CORPUS.is_file() or not COMPOSITION_MANIFEST.is_file():
    raise FileNotFoundError('Attach the reviewed evaluation-disjoint composition corpus and manifest before running this stage.')
composition_preflight_run = subprocess.run(
    [sys.executable, str(runner), '--corpus', str(COMPOSITION_CORPUS), '--manifest', str(COMPOSITION_MANIFEST),
     '--output-dir', str(COMPOSITION_RUN_DIR)], check=True, text=True, capture_output=True)
COMPOSITION_PREFLIGHT = json.loads(composition_preflight_run.stdout)
CAN_COMPOSITION_TRAIN = (COMPOSITION_PREFLIGHT.get('status') == 'READY_TO_TRAIN'
                         and COMPOSITION_PREFLIGHT.get('training_packages_ready') is True
                         and HF_ACCESS_OK and torch.cuda.is_available())
print(composition_preflight_run.stdout)
print('Composition GPU + secret + package gate:', 'READY' if CAN_COMPOSITION_TRAIN else 'BLOCKED')


In [ ]:
if RUN_COMPOSITION_SFT:
    if not CAN_COMPOSITION_TRAIN:
        raise RuntimeError('Composition SFT requires successful GPU, secret/model-access, and corpus preflights.')
    subprocess.run([sys.executable, str(runner), '--corpus', str(COMPOSITION_CORPUS),
                    '--manifest', str(COMPOSITION_MANIFEST), '--output-dir', str(COMPOSITION_RUN_DIR),
                    '--include-splits', 'forget', 'retain', 'train_only', '--train'], check=True)
    comp_run = json.loads((COMPOSITION_RUN_DIR/'run_manifest.json').read_text())
    if comp_run.get('status') != 'TRAINING_COMPLETE' or comp_run.get('training_variant') != 'composition_sft':
        raise RuntimeError(f'Composition SFT did not complete as expected: {comp_run}')
    print('Composition SFT complete:', comp_run['rows'], 'rows; seed', comp_run['seed'])
else:
    print('Composition SFT not started; RUN_COMPOSITION_SFT=False.')


In [ ]:
if RUN_COMPOSITION_RETAIN_ONLY:
    if not CAN_COMPOSITION_TRAIN:
        raise RuntimeError('Composition retain-only SFT requires successful runtime preflights.')
    subprocess.run([sys.executable, str(runner), '--corpus', str(COMPOSITION_CORPUS),
                    '--manifest', str(COMPOSITION_MANIFEST), '--output-dir', str(COMPOSITION_RETAIN_DIR),
                    '--include-splits', 'retain', 'train_only', '--train'], check=True)
    retain_run = json.loads((COMPOSITION_RETAIN_DIR/'run_manifest.json').read_text())
    if retain_run.get('status') != 'TRAINING_COMPLETE' or retain_run.get('training_variant') != 'composition_retain_only':
        raise RuntimeError(f'Composition retain-only SFT did not complete as expected: {retain_run}')
    print('Composition retain-only reference complete:', retain_run['rows'], 'rows')
else:
    print('Composition retain-only training not started.')


In [ ]:
COMPOSITION_BASELINE_GATE = False
COMPOSITION_BASELINE_COUNTS = {}
if RUN_COMPOSITION_BASELINE_EVALUATION:
    if not EVAL_READY:
        raise RuntimeError('Evaluation translation gate is not ready.')
    sft_run_path = COMPOSITION_RUN_DIR / 'run_manifest.json'
    retain_run_path = COMPOSITION_RETAIN_DIR / 'run_manifest.json'
    if not sft_run_path.is_file() or not retain_run_path.is_file():
        raise FileNotFoundError('Composition SFT and retain-only run manifests are required before the baseline gate.')
    sft_run = json.loads(sft_run_path.read_text())
    retain_run = json.loads(retain_run_path.read_text())
    checkpoints = ['--checkpoint', 'pretrained=base', '--checkpoint', f"sft={sft_run['adapter_dir']}",
                   '--checkpoint', f"retain_only={retain_run['adapter_dir']}"]
    evaluator = PROJECT_ROOT / 'scripts' / 'evaluate_crosslingual_unlearning.py'
    subprocess.run([sys.executable, str(evaluator), '--eval-root', str(EVAL_ROOT), '--output-dir', str(COMPOSITION_EVAL_OUT),
                    '--baseline-only', *checkpoints], check=True)
    case_path = COMPOSITION_EVAL_OUT / 'case_metrics.jsonl'
    case_rows = [json.loads(line) for line in case_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    by_key = {(row['instance_id'], row['language']): row for row in case_rows if row['model'] == 'sft'}
    en_ids = {iid for (iid, lang), row in by_key.items() if lang == 'en' and row['split'] == 'forget'
              and row['direct_correct'] and row['multi_correct']}
    target_languages = ('fr', 'de', 'hi', 'th')
    for lang in target_languages:
        lang_ids = {iid for (iid, row_lang), row in by_key.items() if row_lang == lang and row['split'] == 'forget'
                    and row['direct_correct'] and row['multi_correct']}
        paired = en_ids & lang_ids
        groups = {by_key[(iid, lang)]['bootstrap_cluster_id'] for iid in paired}
        COMPOSITION_BASELINE_COUNTS[lang] = {'bilingual_precompetent_cases': len(paired), 'entity_groups': len(groups)}
    # This is a pilot feasibility floor, not a power claim. It prevents spending
    # unlearning GPU time when matched competence is too sparse to interpret.
    COMPOSITION_BASELINE_GATE = all(v['bilingual_precompetent_cases'] >= 15 and v['entity_groups'] >= 8
                                    for v in COMPOSITION_BASELINE_COUNTS.values())
    COMPOSITION_TARGET_IDS.parent.mkdir(parents=True, exist_ok=True)
    target_manifest = {
        'status': 'SFT_PRECOMPETENT_ENGLISH_FORGET_TARGETS',
        'language': 'en',
        'split': 'forget',
        'selection_rule': 'SFT direct and full multi-hop answers both correct in English',
        'selected_source_qa_ids': sorted(en_ids),
        'baseline_gate_status': 'PASS' if COMPOSITION_BASELINE_GATE else 'FAIL',
        'pilot_gate_thresholds': {
            'minimum_matched_cases_per_language': 15,
            'minimum_entity_groups_per_language': 8,
        },
        'bilingual_counts_by_language': COMPOSITION_BASELINE_COUNTS,
        'source_corpus_sha256': hashlib.sha256(COMPOSITION_CORPUS.read_bytes()).hexdigest(),
        'sft_run_manifest_sha256': hashlib.sha256(sft_run_path.read_bytes()).hexdigest(),
        'sft_adapter_dir': sft_run['adapter_dir'],
        'case_metrics_path': str(case_path.resolve()),
        'case_metrics_sha256': hashlib.sha256(case_path.read_bytes()).hexdigest(),
        'human_reviewed': False,
        'translation_review_mode': 'AI_ONLY',
    }
    COMPOSITION_TARGET_IDS.write_text(json.dumps(target_manifest, indent=2) + '\n', encoding='utf-8')
    print('Forget-split bilingual precompetence:', COMPOSITION_BASELINE_COUNTS)
    print('Pre-unlearning pilot gate (>=15 matched cases and >=8 groups in each target language):',
          'PASS' if COMPOSITION_BASELINE_GATE else 'FAIL — stop before unlearning')
else:
    print('Composition baseline evaluation not started; unlearning gate remains closed.')


In [ ]:
COMPOSITION_UNLEARN_RUNS = {}
if RUN_COMPOSITION_UNLEARNING:
    if not CAN_COMPOSITION_TRAIN:
        raise RuntimeError('Composition unlearning requires successful runtime preflights.')
    if not COMPOSITION_BASELINE_GATE:
        raise RuntimeError('Pre-unlearning bilingual competence gate failed. Review baseline diagnostics; do not run unlearning.')
    sft_run_path = COMPOSITION_RUN_DIR / 'run_manifest.json'
    if not sft_run_path.is_file():
        raise FileNotFoundError('Complete composition SFT before unlearning.')
    if not COMPOSITION_TARGET_IDS.is_file():
        raise FileNotFoundError('Run the baseline-only evaluation first to create the competence target manifest.')
    target_manifest = json.loads(COMPOSITION_TARGET_IDS.read_text())
    expected_forget_rows = len(target_manifest['selected_source_qa_ids'])
    if target_manifest.get('baseline_gate_status') != 'PASS' or expected_forget_rows == 0:
        raise RuntimeError('The baseline target manifest is empty or its feasibility gate failed.')
    unlearn_script = PROJECT_ROOT / 'scripts' / 'unlearn_lora.py'
    preflight_reports = {}
    # Each target is SFT-precompetent in English; the manifest is hash-bound to
    # this corpus, adapter and case-metrics file. The CLI independently verifies it.
    for method in METHODS:
        out = COMPOSITION_UNLEARN_ROOT / method
        check = subprocess.run([sys.executable, str(unlearn_script), '--corpus', str(COMPOSITION_CORPUS),
                                '--corpus-manifest', str(COMPOSITION_MANIFEST), '--sft-run-manifest', str(sft_run_path),
                                '--target-ids-manifest', str(COMPOSITION_TARGET_IDS),
                                '--output-dir', str(out), '--method', method],
                               check=True, text=True, capture_output=True)
        report = json.loads(check.stdout)
        print(method.upper(), 'preflight:', report.get('status'), report.get('errors', []))
        if report.get('status') != 'READY_TO_UNLEARN':
            raise RuntimeError(f'{method.upper()} composition unlearning preflight failed: {report}')
        if report.get('english_forget_rows') != expected_forget_rows or report.get('holdout_rows_used') != 0:
            raise RuntimeError(f'{method.upper()} preflight selected unexpected forget/holdout rows: {report}')
        preflight_reports[method] = report
    for method in METHODS:
        out = COMPOSITION_UNLEARN_ROOT / method
        subprocess.run([sys.executable, str(unlearn_script), '--corpus', str(COMPOSITION_CORPUS),
                        '--corpus-manifest', str(COMPOSITION_MANIFEST), '--sft-run-manifest', str(sft_run_path),
                        '--target-ids-manifest', str(COMPOSITION_TARGET_IDS),
                        '--output-dir', str(out), '--method', method, '--train'], check=True)
        run = json.loads((out / 'run_manifest.json').read_text())
        expected_retain_rows = preflight_reports[method]['english_retain_rows']
        if (run.get('status') != 'UNLEARNING_COMPLETE' or run.get('holdout_rows_used') != 0
                or run.get('english_forget_rows') != expected_forget_rows
                or run.get('english_retain_rows') != expected_retain_rows
                or run.get('english_forget_target_ids') != target_manifest['selected_source_qa_ids']
                or run.get('source_corpus_sha256') != hashlib.sha256(COMPOSITION_CORPUS.read_bytes()).hexdigest()
                or run.get('target_ids_manifest_sha256') != hashlib.sha256(COMPOSITION_TARGET_IDS.read_bytes()).hexdigest()):
            raise RuntimeError(f'{method.upper()} composition unlearning output failed validation: {run}')
        COMPOSITION_UNLEARN_RUNS[method] = out / 'run_manifest.json'
        print(method.upper(), 'complete; English r1 rows:', run['english_forget_rows'], 'precompetent forget +',
              run['english_retain_rows'], 'retain; holdout rows used:', run['holdout_rows_used'])
else:
    print('Composition unlearning not started.')


In [ ]:
# Exploratory follow-up only: preserves the failed baseline gate and labels results underpowered.
# Set True only after reviewing the baseline diagnostic report.
RUN_LOW_POWER_EXPLORATORY_UNLEARNING = False
if RUN_LOW_POWER_EXPLORATORY_UNLEARNING:
    import importlib.util
    import json, hashlib, re
    from pathlib import Path

    unlearn_script = PROJECT_ROOT / 'scripts' / 'unlearn_lora.py'
    source_text = unlearn_script.read_text(encoding='utf-8')
    hop_pattern = re.compile(r"(?m)^    supported_corpus_hops = \{.*?^    \}", re.S)
    replacement = '\\n'.join([
        '    supported_corpus_hops = {',
        '        "r1", "r2", "composition_2hop", "composition_3hop",',
        '        "composition_support_q1", "composition_support_q1_all",',
        '        "composition_support_q2", "composition_support_q3",',
        '    }',
    ])
    patched_source, patch_count = hop_pattern.subn(lambda _: replacement, source_text)
    if patch_count != 1:
        raise RuntimeError(f'Expected one supported-hop allowlist in unlearning script; found {patch_count}.')
    patched_script = Path('/kaggle/working/unlearn_lora_v4_exploratory_fixed.py')
    patched_script.write_text(patched_source, encoding='utf-8')
    spec = importlib.util.spec_from_file_location('unlearn_lora_exploratory_fixed', patched_script)
    unlearn_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(unlearn_mod)

    target_manifest = json.loads(COMPOSITION_TARGET_IDS.read_text(encoding='utf-8'))
    if target_manifest.get('baseline_gate_status') != 'FAIL':
        raise RuntimeError('This low-power path is only for a target manifest that records baseline_gate_status=FAIL.')
    gate_errors = {
        'The pre-unlearning bilingual competence feasibility gate did not pass',
        'Target manifest lacks a passing matched-case/entity-group count for every target language',
    }
    if not target_manifest.get('selected_source_qa_ids'):
        raise RuntimeError('No SFT-precompetent English forget targets are available.')

    reports = {}
    for method in ('ga', 'gd', 'npo'):
        report = unlearn_mod.preflight(
            COMPOSITION_CORPUS, COMPOSITION_MANIFEST,
            COMPOSITION_RUN_DIR / 'run_manifest.json', COMPOSITION_TARGET_IDS, method,
        )
        print(method.upper(), 'exploratory preflight:', report.get('status'), report.get('errors'))
        if report.get('status') != 'BLOCKED' or set(report.get('errors', [])) != gate_errors:
            raise RuntimeError(f'{method.upper()} has blockers beyond the failed feasibility floor; refusing training: {report}')
        if report.get('holdout_rows_used') != 0 or report.get('english_forget_rows') != len(target_manifest['selected_source_qa_ids']):
            raise RuntimeError(f'{method.upper()} selected unexpected target or holdout rows: {report}')
        reports[method] = report

    print('All three method preflights passed apart from the declared low-power gate; no other blocker remains.')
    print('This is exploratory only. The baseline feasibility gate remains FAIL.')

    COMPOSITION_UNLEARN_RUNS = {}
    for method in ('ga', 'gd', 'npo'):
        out = COMPOSITION_UNLEARN_ROOT / method
        run_path = out / 'run_manifest.json'
        if run_path.is_file():
            result = json.loads(run_path.read_text(encoding='utf-8'))
            if (result.get('source_corpus_sha256') != hashlib.sha256(COMPOSITION_CORPUS.read_bytes()).hexdigest()
                    or result.get('target_ids_manifest_sha256') != hashlib.sha256(COMPOSITION_TARGET_IDS.read_bytes()).hexdigest()
                    or result.get('run_mode') != 'EXPLORATORY_LOW_POWER_AFTER_FAILED_GATE'):
                raise RuntimeError(f'{method.upper()} output contains a different prior run; refusing overwrite.')
            print(method.upper(), 'matching exploratory result reused.')
        else:
            result = unlearn_mod.train(
                COMPOSITION_CORPUS, COMPOSITION_MANIFEST,
                COMPOSITION_RUN_DIR / 'run_manifest.json', COMPOSITION_TARGET_IDS,
                out, method, epochs=3, lr=1e-5, retain_weight=1.0,
                beta=0.1, max_length=512, seed=42,
            )
            result['run_mode'] = 'EXPLORATORY_LOW_POWER_AFTER_FAILED_GATE'
            result['baseline_gate_status'] = target_manifest['baseline_gate_status']
            result['baseline_gate_thresholds'] = target_manifest['pilot_gate_thresholds']
            result['baseline_bilingual_counts_by_language'] = target_manifest['bilingual_counts_by_language']
            result['patched_unlearning_script_sha256'] = hashlib.sha256(patched_script.read_bytes()).hexdigest()
            result['interpretation_limit'] = 'Underpowered exploratory pilot; not confirmatory. Baseline gate remains FAIL.'
            run_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
        expected_forget = len(target_manifest['selected_source_qa_ids'])
        if (result.get('status') != 'UNLEARNING_COMPLETE'
                or result.get('holdout_rows_used') != 0
                or result.get('english_forget_rows') != expected_forget
                or result.get('english_retain_rows') != reports[method]['english_retain_rows']
                or result.get('english_forget_target_ids') != sorted(target_manifest['selected_source_qa_ids'])
                or not (Path(result.get('adapter_dir', '')) / 'adapter_config.json').is_file()):
            raise RuntimeError(f'{method.upper()} exploratory output failed integrity checks: {result}')
        COMPOSITION_UNLEARN_RUNS[method] = run_path
        print(method.upper(), 'complete:', result['optimizer_steps'], 'steps;',
              result['english_forget_rows'], 'English forget rows;', result['english_retain_rows'],
              'retain rows; holdout rows used:', result['holdout_rows_used'])


In [ ]:
if RUN_COMPOSITION_POST_EVALUATION:
    if not EVAL_READY:
        raise RuntimeError('Evaluation translation gate is not ready.')
    sft_run_path = COMPOSITION_RUN_DIR / 'run_manifest.json'
    retain_run_path = COMPOSITION_RETAIN_DIR / 'run_manifest.json'
    if not sft_run_path.is_file() or not retain_run_path.is_file():
        raise FileNotFoundError('Composition SFT and retain-only run manifests are required.')
    sft_run = json.loads(sft_run_path.read_text())
    retain_run = json.loads(retain_run_path.read_text())
    checkpoints = ['--checkpoint', 'pretrained=base', '--checkpoint', f"sft={sft_run['adapter_dir']}",
                   '--checkpoint', f"retain_only={retain_run['adapter_dir']}"]
    for method in METHODS:
        run_path = COMPOSITION_UNLEARN_ROOT / method / 'run_manifest.json'
        if not run_path.is_file():
            raise FileNotFoundError(f'Missing composition unlearning manifest for {method}.')
        run = json.loads(run_path.read_text())
        checkpoints += ['--checkpoint', f"{method.upper()}={run['adapter_dir']}"]
    evaluator = PROJECT_ROOT / 'scripts' / 'evaluate_crosslingual_unlearning.py'
    subprocess.run([sys.executable, str(evaluator), '--eval-root', str(EVAL_ROOT), '--output-dir', str(COMPOSITION_EVAL_OUT),
                    '--contrast-model', 'GA', '--contrast-model', 'GD', '--contrast-model', 'NPO', *checkpoints], check=True)
    print('Composition diagnostic report:', COMPOSITION_EVAL_OUT/'diagnostic_report.md')
    print('Forget multi-hop prompts are held out, but their atomic support facts were trained; this measures composition generalization over seen facts. Retain composition examples overlap training by design.')
else:
    print('Composition evaluation not started.')


### Composition follow-up interpretation

The fixed evaluation split is unchanged. Type-B practice chains are source-fact verified and disjoint from evaluation entities, answers, exact questions, and path facts. Run the baseline first. The notebook blocks unlearning unless every target language has at least 15 matched bilingual precompetent forget cases across at least 8 entity groups; this is a pilot feasibility floor, not a power claim. Retain scores are training-overlap controls. Translation checks are automated AI-only consistency checks; no human fluency review was performed.